# 07 · Ablation

Evaluates the cumulative contribution of each component on the Mendeley-held-out
fold: (A) baseline, (B) source-balanced sampling, (C) noise-aware loss,
(D) curriculum with label-quality weighting, (E) domain-adversarial training, plus (F) direct 5-class and (G) ordinal/CORAL for RQ3 and the ordinal comparison.
A single seed is used for screening; the selected configuration is confirmed at
full seed count in the leave-one-dataset-out folds.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


In [ ]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)
print('Split helper ready.')

Split helper ready.


## Execution

In [ ]:
import time
test_ds=config.LODO_FOLDS[config.ABLATION_FOLD]
results=[]
for name,mech in config.ABLATION_CONFIGS.items():
    run=f'ablation_{name}_seed0'
    tr,va,te=make_splits(manifest,test_ds,seed=0)
    print(f'--- {run} (train={len(tr):,} val={len(va):,} test={len(te):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,0,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
print(df[['run_name','internal_acc5','external_acc5','external_qwk','gap']].to_string(index=False))
df.to_csv(str(config.RESULTS_DIR/'ablation_summary.csv'),index=False)

print('\n=== Extra configs: F_direct5class (RQ3), G_ordinal (CORAL) ===')
for name,mech in config.EXTRA_CONFIGS.items():
    run=f'ablation_{name}_seed0'
    tr,va,te=make_splits(manifest,test_ds,seed=0)
    print(f'--- {run} (train={len(tr):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,0,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
df.to_csv(str(config.RESULTS_DIR/'ablation_summary.csv'),index=False)
print('\nF vs D answers RQ3 (hierarchical vs direct 5-class).')
print('G vs D answers the ordinal/CORAL comparison (expose 2.1).')

--- ablation_A_naive_seed0 (train=45,304 val=7,995 test=8,259) ---
  [ablation_A_naive_seed0] complete - skip.
--- ablation_B_sampler_seed0 (train=45,304 val=7,995 test=8,259) ---
  [ablation_B_sampler_seed0] complete - skip.
--- ablation_C_noiseloss_seed0 (train=45,304 val=7,995 test=8,259) ---
  [ablation_C_noiseloss_seed0] complete - skip.
--- ablation_D_curriculum_seed0 (train=45,304 val=7,995 test=8,259) ---
  [ablation_D_curriculum_seed0] complete - skip.
--- ablation_E_domain_adv_seed0 (train=45,304 val=7,995 test=8,259) ---
  [ablation_E_domain_adv_seed0] complete - skip.
                   run_name  internal_acc5  external_acc5  external_qwk      gap
     ablation_A_naive_seed0       0.581989       0.273762      0.231676 0.308227
   ablation_B_sampler_seed0       0.566729       0.333575      0.343899 0.233154
 ablation_C_noiseloss_seed0       0.559850       0.345320      0.370454 0.214530
ablation_D_curriculum_seed0       0.542964       0.362877      0.381448 0.180087
ablation

## Configuration E and G — Stabilization and Correction

Initial training of configuration E (domain-adversarial) and configuration G
(ordinal) exhibited optimization failures. In configuration E, the loss diverged
once the gradient-reversal weight exceeded approximately 0.7, causing the model to
collapse to a single-class prediction. In configuration G, the cumulative-logit
ordinal formulation failed to converge, yielding near-random accuracy and negative
agreement.

Two corrections were applied. First, gradient-norm clipping (maximum norm 1.0) was
introduced to bound the combined task and adversarial gradients, preventing the
divergence characteristic of domain-adversarial training. The peak adversarial
weight was additionally reduced from 1.0 to 0.3, with the sweep range adjusted to
{0.1, 0.3, 0.5}. Second, the ordinal objective was reformulated as a
distance-weighted cross-entropy that scales the per-sample loss by the deviation
between the expected and true rank, preserving ordinality while remaining
compatible with the standard prediction path.

Both configurations were retrained under these corrections; the remaining
configurations were unaffected and are reused from the prior run.